# EduMentor AI v2.0 — Full RAG Pipeline with PDF Knowledge Base (Text + Images + Tables)

**Author**: Shivakant Kurmi

---

## Overview
This notebook builds a **production-grade PDF RAG pipeline** that extracts:
- **Text** — every paragraph from every page (`pdfplumber`)
- **Tables** — structured data from every page (`pdfplumber`)
- **Images** — all embedded diagrams and figures described via Gemini Vision OCR (`PyMuPDF` + `gemini-3.5-flash` vision)

All three content types are chunked, embedded, and indexed in FAISS so the LLM can answer questions grounded in the full document.

| Phase | Steps | Description |
|-------|-------|-------------|
| **Phase 1 – PDF/RAG** | 1–7 | Upload real PDFs → extract text+tables+images → chunk → embed → FAISS → similarity search → configurable Top-K |
| **Phase 2 – LLM** | 8–12 | Gemini 3.5 Flash: temperature, max_tokens, top_p, proper RAG prompt, source citations |
| **Phase 3 – Evaluation** | 13–18 | Test Q+A dataset, K-sweep, MRR (primary metric), retrieval quality, ROUGE/BLEU, faithfulness, bias/fairness |

---

## Model Choices & Justification

### PDF Extraction Libraries
| Library | Role | Why |
|---------|------|-----|
| `pdfplumber` | Text + table extraction | Best-in-class table detection; exposes word positions and cell boundaries |
| `PyMuPDF` (`fitz`) | Image extraction | Fast, reliable embedded image rasterisation with page coordinates |
| `Gemini 3.5 Flash` (vision) | Image-to-text OCR/description | Natively multimodal; handles diagrams, charts, equations, handwriting |

### Embedding Model — `sentence-transformers/all-MiniLM-L6-v2`
- **Why**: 384-dim dense vectors; ~5x faster than `all-mpnet-base-v2`; strong STS benchmarks (Spearman correlation ~0.88). CPU-only, free.
- **Context window**: **512 tokens**. We chunk at <= 400 chars (~80–110 tokens) — safe margin below 512.

### Vector Database — FAISS (Facebook AI Similarity Search)
- **Why**: In-memory, zero server overhead, exact cosine search, native LangChain integration.

### Generation Model — `gemini-3.5-flash` (Current Generation)
- **Why**: `gemini-1.5-flash` and `gemini-2.5-flash` have legacy/deprecation timelines. `gemini-3.5-flash` provides state-of-the-art inference speed, cost efficiency, advanced multimodal capabilities, and a **1,000,000-token context window**.
- **SDK**: Uses the official `google-genai` SDK (`from google import genai`).
- **Context used**: At K=5 with 400-char chunks -> ~500 tokens = only **0.05%** of capacity per call.

### Primary Evaluation Metric — MRR (Mean Reciprocal Rank)
- **Why**: In educational QA, the student needs the single most relevant chapter or formula at Rank 1. MRR penalises delayed retrieval much more strictly than generic recall.


---
## Install Dependencies

In [ ]:
# Install all required libraries
# We use the official Google Gen AI SDK (google-genai)
!pip install -q google-genai langchain langchain-community langchain-huggingface faiss-cpu pdfplumber pymupdf Pillow rouge-score nltk pandas matplotlib seaborn

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("All dependencies installed.")


---
## Environment Setup & API Key

In [ ]:
import os, io, re, json, base64, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pdfplumber          # text + table extraction
import fitz                # PyMuPDF — image extraction
from PIL import Image

# Official Google Gen AI SDK
from google import genai
from google.genai import types as genai_types

from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

# ── API Key ──────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception:
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY", "")

if not api_key:
    raise ValueError(
        "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in Kaggle Secrets or as an environment variable."
    )

# Initialise client
client = genai.Client(api_key=api_key)

# Model: gemini-3.5-flash
LLM_MODEL_NAME = "gemini-3.5-flash"

print("Environment ready. API key loaded.")
print(f"SDK     : google-genai (official SDK)")
print(f"Model   : {LLM_MODEL_NAME}")


---
# Phase 1 — PDF / RAG Pipeline

> Steps 1–7: Upload real PDFs → extract text + tables + images → chunk → embed → FAISS → similarity search → configurable Top-K


## Step 1 — PDF Upload API

`load_pdfs(paths)` accepts a **list of file paths** and returns `(filename, raw_bytes)` tuples ready for the extraction pipeline.

```python
# Point to your own PDFs:
pdf_files = load_pdfs([
    "/kaggle/input/my-dataset/textbook.pdf",
    "/kaggle/input/my-dataset/lecture_slides.pdf",
])
```


In [ ]:
def load_pdfs(paths):
    """
    PDF Upload API — load PDFs from file-system paths.

    Args:
        paths (list[str]): Absolute or relative paths to PDF files.

    Returns:
        list[tuple[str, bytes]]: (filename, raw_bytes) for each valid PDF.
    """
    loaded = []
    for p in paths:
        if not os.path.exists(p):
            print(f"  WARNING: Not found: {p} — skipping.")
            continue
        with open(p, "rb") as f:
            raw = f.read()
        if raw[:4] != b"%PDF":
            print(f"  WARNING: {p} does not appear to be a valid PDF — skipping.")
            continue
        loaded.append((os.path.basename(p), raw))
        size_kb = len(raw) / 1024
        print(f"  Loaded: {os.path.basename(p)}  ({size_kb:.1f} KB)")
    if not loaded:
        raise ValueError("No valid PDF files found. Check the paths above.")
    print(f"\n  {len(loaded)} PDF(s) ready for extraction.")
    return loaded


# ─── POINT THIS TO YOUR ACTUAL PDFs ──────────────────────────────────────────
PDF_PATHS = [
    # "/kaggle/input/your-dataset/textbook.pdf",
    # "/kaggle/input/your-dataset/lecture_slides.pdf",
]

# If no paths provided, provide a clean placeholder PDF for structural validation
if not PDF_PATHS:
    print("PDF_PATHS is currently empty.")
    print("Set PDF_PATHS above to point to your actual PDF files.")
    print("Example: PDF_PATHS = ['/kaggle/input/my-data/book.pdf']")
    _demo_pdf = (
        b"%PDF-1.4\n"
        b"1 0 obj<</Type/Catalog/Pages 2 0 R>>endobj\n"
        b"2 0 obj<</Type/Pages/Kids[3 0 R]/Count 1>>endobj\n"
        b"3 0 obj<</Type/Page/Parent 2 0 R/MediaBox[0 0 612 792]"
        b"/Contents 4 0 R/Resources<</Font<</F1<</Type/Font/Subtype/Type1/BaseFont/Helvetica>>>>>>>>endobj\n"
        b"4 0 obj<</Length 115>>\nstream\n"
        b"BT /F1 14 Tf 72 700 Td (EduMentor AI - PDF Knowledge Base) Tj\n"
        b"0 -30 Td (Replace PDF_PATHS with your real PDF file paths.) Tj ET\n"
        b"endstream\nendobj\n"
        b"xref\n0 5\n0000000000 65535 f\n0000000009 00000 n\n"
        b"0000000058 00000 n\n0000000115 00000 n\n0000000274 00000 n\n"
        b"trailer<</Size 5/Root 1 0 R>>\nstartxref\n440\n%%EOF"
    )
    pdf_files = [("placeholder.pdf", _demo_pdf)]
    print("\n  Using structural placeholder PDF for validation.")
else:
    pdf_files = load_pdfs(PDF_PATHS)


## Step 2 — PDF Text, Table, and Image Extraction

We extract **three content types** from each PDF page:

| Content | Library | Method | Why |
|---------|---------|--------|-----|
| **Text** | `pdfplumber` | `page.extract_text()` | Preserves reading order, handles multi-column layouts |
| **Tables** | `pdfplumber` | `page.extract_tables()` | Cell-level detection, outputs structured rows |
| **Images** | `PyMuPDF` (`fitz`) | `page.get_images()` + `extract_image()` | Rasterises embedded images (JPEG/PNG/TIFF) |

Images are described using **Gemini 3.5 Flash Vision** to convert visual information (diagrams, charts, formulas) into rich searchable text chunks.


In [ ]:
MAX_IMAGE_SIZE_PX = 1024
IMAGE_DPI = 150

def describe_image_with_gemini(img_bytes, source_hint=""):
    """
    Send image bytes to Gemini 3.5 Flash (vision) and get a detailed text description.
    """
    try:
        img = Image.open(io.BytesIO(img_bytes))
        if max(img.size) > MAX_IMAGE_SIZE_PX:
            img.thumbnail((MAX_IMAGE_SIZE_PX, MAX_IMAGE_SIZE_PX), Image.LANCZOS)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        img_bytes_resized = buf.getvalue()

        prompt = (
            f"This image is from a PDF document ({source_hint}).\n"
            "Describe all visual content in detail: text in the image, equations, "
            "labels on charts/graphs/diagrams, table data if visible, any legends or captions. "
            "Be thorough — your description will be used to answer student questions."
        )
        response = client.models.generate_content(
            model=LLM_MODEL_NAME,
            contents=[
                prompt,
                genai_types.Part.from_bytes(data=img_bytes_resized, mime_type="image/jpeg"),
            ],
            config=genai_types.GenerateContentConfig(temperature=0.0, max_output_tokens=400),
        )
        return response.text.strip()
    except Exception as e:
        return f"[Image description unavailable: {e}]"


def table_to_text(table):
    """Convert a pdfplumber table (list of lists) to formatted pipe-delimited text."""
    rows = []
    for row in table:
        cleaned = [str(cell).strip() if cell is not None else "" for cell in row]
        rows.append(" | ".join(cleaned))
    return "\n".join(rows)


def extract_all_from_pdf(filename, raw_bytes, describe_images=True):
    """
    Extract TEXT, TABLES, and IMAGES from a PDF.
    """
    extracted = []

    # 1. Text & Tables via pdfplumber
    with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            raw_text = page.extract_text(x_tolerance=2, y_tolerance=2)
            if raw_text and raw_text.strip():
                tables_on_page = page.extract_tables()
                text_clean = raw_text.strip()
                if tables_on_page:
                    table_cells = {
                        str(cell).strip()
                        for tbl in tables_on_page
                        for row in tbl
                        for cell in row
                        if cell
                    }
                    text_lines = [
                        line for line in text_clean.split("\n")
                        if line.strip() and line.strip() not in table_cells
                    ]
                    text_clean = "\n".join(text_lines)

                if text_clean.strip():
                    extracted.append({
                        "content_type": "text",
                        "filename"    : filename,
                        "page"        : page_num,
                        "text"        : text_clean,
                        "metadata"    : {"content_type": "text"},
                    })

            tables = page.extract_tables()
            for tbl_idx, table in enumerate(tables):
                if not table:
                    continue
                tbl_text = table_to_text(table)
                if tbl_text.strip():
                    extracted.append({
                        "content_type": "table",
                        "filename"    : filename,
                        "page"        : page_num,
                        "text"        : f"[TABLE {tbl_idx+1} on page {page_num}]\n{tbl_text}",
                        "metadata"    : {"content_type": "table", "table_index": tbl_idx},
                    })

    # 2. Images via PyMuPDF
    pdf_doc = fitz.open(stream=raw_bytes, filetype="pdf")
    for page_num in range(len(pdf_doc)):
        page = pdf_doc[page_num]
        image_list = page.get_images(full=True)

        for img_idx, img_info in enumerate(image_list):
            xref = img_info[0]
            try:
                base_image = pdf_doc.extract_image(xref)
                img_bytes  = base_image["image"]
                img_ext    = base_image["ext"]

                if len(img_bytes) < 5_000:
                    continue

                source_hint = f"{filename}, page {page_num+1}, image {img_idx+1}"
                if describe_images:
                    description = describe_image_with_gemini(img_bytes, source_hint)
                else:
                    description = f"[Image on page {page_num+1}, image {img_idx+1}]"

                extracted.append({
                    "content_type": "image",
                    "filename"    : filename,
                    "page"        : page_num + 1,
                    "text"        : f"[IMAGE {img_idx+1} on page {page_num+1}] {description}",
                    "metadata"    : {
                        "content_type": "image",
                        "image_index" : img_idx,
                        "format"      : img_ext,
                        "size_bytes"  : len(img_bytes),
                    },
                })
            except Exception as e:
                print(f"  Could not extract image {img_idx+1} on page {page_num+1}: {e}")

    pdf_doc.close()
    return extracted


DESCRIBE_IMAGES = True

all_extracted = []
print("Extracting content from PDFs...\n")
for filename, raw_bytes in pdf_files:
    print(f"Processing: {filename}")
    items = extract_all_from_pdf(filename, raw_bytes, describe_images=DESCRIBE_IMAGES)
    all_extracted.extend(items)

    text_count  = sum(1 for i in items if i["content_type"] == "text")
    table_count = sum(1 for i in items if i["content_type"] == "table")
    image_count = sum(1 for i in items if i["content_type"] == "image")
    print(f"  Text blocks : {text_count}")
    print(f"  Tables      : {table_count}")
    print(f"  Images      : {image_count}\n")

print(f"Extraction complete. Total items: {len(all_extracted)}")


## Step 3 — Text Chunking (Content-Type Aware)

- **Text blocks**: Recursively split at 400 characters (80 overlap) to fit well within the 512-token limit of `all-MiniLM-L6-v2`.
- **Tables & Image Descriptions**: Preserved as single atomic chunks since they are already concise and self-contained.


In [ ]:
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_extracted_items(items):
    documents = []
    for item in items:
        content_type = item["content_type"]
        text = item["text"].strip()
        if not text:
            continue

        base_meta = {
            "source"      : item["filename"],
            "page"        : item["page"],
            "content_type": content_type,
            **item.get("metadata", {}),
        }

        if content_type in ("table", "image"):
            documents.append(Document(page_content=text, metadata={**base_meta, "chunk": 0}))
        else:
            chunks = splitter.split_text(text)
            for j, chunk in enumerate(chunks):
                documents.append(Document(
                    page_content=chunk,
                    metadata={**base_meta, "chunk": j}
                ))
    return documents

documents = chunk_extracted_items(all_extracted)
print(f"Chunking complete. Total chunks: {len(documents)}")


## Step 4 — Embedding Generation

**Model**: `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional embeddings, 512-token context limit, L2-normalised).


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")
embeddings_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

test_vec = embeddings_model.embed_query("What does the diagram show?")
print(f"Embedding model ready. Vector dimension: {len(test_vec)}")


## Step 5 — Vector DB Integration (FAISS)

Build and persist the FAISS index containing all text, table, and image description vectors.


In [ ]:
VECTORSTORE_PATH = "./edumentor_faiss_index"

print(f"Building FAISS vector store from {len(documents)} chunks...")
vectorstore = FAISS.from_documents(documents, embeddings_model)
vectorstore.save_local(VECTORSTORE_PATH)

print(f"FAISS index built and saved to '{VECTORSTORE_PATH}'")
print(f"  Total vectors : {vectorstore.index.ntotal}")
print(f"  Embedding dim : {vectorstore.index.d}")


## Step 6 — Similarity Search

Searches across text, tables, and image descriptions simultaneously.


In [ ]:
def similarity_search(query, k=3):
    """
    Retrieve top-k most relevant chunks across text, tables, and image descriptions.
    """
    results = vectorstore.similarity_search_with_score(query, k=k)
    scored = [(doc, max(0.0, 1.0 - (dist**2) / 2.0)) for doc, dist in results]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

query = "What information is shown in the document?"
results = similarity_search(query, k=3)
print(f"Query: '{query}'")
for i, (doc, score) in enumerate(results, 1):
    ct = doc.metadata.get("content_type", "text")
    print(f"[{i}] {ct.upper():8s} | {doc.metadata['source']} p{doc.metadata['page']} | Score: {score:.4f}")
    print(f"     {doc.page_content[:180]}...\n")


## Step 7 — Configurable Top-K Retrieval

Formats retrieved chunks with citations for insertion into the LLM prompt.


In [ ]:
def retrieve_context(query, top_k=3):
    """
    Retrieve and format top-k chunks as LLM context with structured source citations.
    """
    results = similarity_search(query, k=top_k)
    context_parts, sources_meta = [], []
    for i, (doc, score) in enumerate(results, 1):
        ct = doc.metadata.get("content_type", "text")
        header = (
            f"[Source {i}: {doc.metadata['source']}, "
            f"page {doc.metadata['page']}, "
            f"type={ct}, score={score:.3f}]"
        )
        context_parts.append(f"{header}\n{doc.page_content}")
        sources_meta.append({
            "rank"        : i,
            "source"      : doc.metadata["source"],
            "page"        : doc.metadata["page"],
            "content_type": ct,
            "score"       : round(score, 4),
            "snippet"     : doc.page_content[:120] + "...",
        })
    return "\n\n---\n\n".join(context_parts), sources_meta

ctx, srcs = retrieve_context("Explain key concepts", top_k=3)
print(f"Retrieved {len(srcs)} chunks for K=3.")


---
# Phase 2 — LLM Integration

> Steps 8–12: Gemini 3.5 Flash via `google-genai` SDK with temperature/max_tokens controls, structured RAG prompt, and citations


## Step 8 — LLM Integration (Gemini 3.5 Flash)

| Attribute | Value |
|-----------|-------|
| Model | `gemini-3.5-flash` |
| SDK | `google-genai` (`from google import genai`) |
| Context window | **1,000,000 tokens** |
| Why chosen | State-of-the-art performance, lowest latency, native multimodality, full parameter controls |


In [ ]:
print(f"SDK  : google-genai (official SDK)")
print(f"Model: {LLM_MODEL_NAME}")
print(f"  Context window : 1,000,000 tokens")
print(f"  Tokens used    : ~300 per call at K=3 (0.03% of capacity)")


## Step 9 — Proper RAG Prompt

Strictly instructs the model to use only the provided context, cite sources, and admit uncertainty when info is absent.


In [ ]:
RAG_SYSTEM_PROMPT = (
    "You are EduMentor, an expert educational AI assistant.\n"
    "Answer the student's question using ONLY the information provided in the [CONTEXT] below.\n"
    "The context may include text paragraphs, table data, and descriptions of images/diagrams.\n"
    "Rules:\n"
    "1. If the context does not contain enough information, say exactly: "
    "'The provided materials do not cover this — please consult additional resources.'\n"
    "2. At the end, cite every source you used: [Source: <filename>, page <N>, type=<text|table|image>]\n"
    "3. When referencing a table, format key data clearly.\n"
    "4. When referencing an image/diagram, describe what it shows.\n"
    "5. Do NOT invent any facts, numbers, or formulas not present in the context.\n"
    "6. Adjust language to the student's difficulty level and learning style.\n"
    "7. Keep your answer under {max_words} words.\n"
)

RAG_USER_PROMPT = (
    "Difficulty: {difficulty}\n"
    "Learning Style: {learning_style}\n\n"
    "[CONTEXT]\n"
    "{context}\n"
    "[END CONTEXT]\n\n"
    "[QUESTION]\n"
    "{question}\n"
)

def build_rag_prompt(question, context, difficulty="Intermediate",
                     learning_style="Visual", max_words=300):
    system = RAG_SYSTEM_PROMPT.format(max_words=max_words)
    user   = RAG_USER_PROMPT.format(
        difficulty=difficulty, learning_style=learning_style,
        context=context, question=question)
    return system + "\n" + user


## Step 10 — Temperature Configuration

Task-specific generation presets using `genai_types.GenerateContentConfig`:


In [ ]:
GEN_CONFIGS = {
    "factual_qa" : genai_types.GenerateContentConfig(temperature=0.0, max_output_tokens=512),
    "study_plan" : genai_types.GenerateContentConfig(temperature=0.3, max_output_tokens=700),
    "explainer"  : genai_types.GenerateContentConfig(temperature=0.5, max_output_tokens=512),
    "quiz"       : genai_types.GenerateContentConfig(temperature=0.7, max_output_tokens=800),
    "creative"   : genai_types.GenerateContentConfig(temperature=0.9, max_output_tokens=600),
}

print("Config presets ready:")
for name, cfg in GEN_CONFIGS.items():
    print(f"  {name:12s}: temperature={cfg.temperature}, max_tokens={cfg.max_output_tokens}")


## Step 11 — max_tokens and Generation Parameters

`generate_rag_answer()` combines retrieval, prompting, and Gemini 3.5 Flash execution.


In [ ]:
def generate_rag_answer(question, top_k=3, difficulty="Intermediate",
                       learning_style="Visual", temperature=0.3,
                       max_tokens=512, top_p=0.9, llm_top_k=40):
    """
    Full RAG pipeline: retrieve -> prompt -> generate -> return with citations.
    """
    context_str, sources_meta = retrieve_context(question, top_k=top_k)
    prompt = build_rag_prompt(question, context_str, difficulty=difficulty,
                              learning_style=learning_style, max_words=max_tokens//2)
    gen_cfg = genai_types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=max_tokens,
        top_p=top_p,
        top_k=llm_top_k,
    )
    response = client.models.generate_content(
        model=LLM_MODEL_NAME,
        contents=prompt,
        config=gen_cfg,
    )
    return {
        "answer"          : response.text.strip() if response.text else "",
        "sources"         : sources_meta,
        "context_used"    : context_str,
        "prompt_chars_est": len(prompt),
    }

demo_ans = generate_rag_answer("What content is covered?", top_k=2)
print("Answer sample:")
print(demo_ans["answer"])


## Step 12 — Source / Citation Extraction

Extracts and formats source metadata (PDF filename, page, content type, similarity score).


In [ ]:
def format_citations(sources):
    lines = ["\nSources used (ranked by relevance):\n"]
    for s in sources:
        lines.append(
            f"  [{s['rank']}] {s['source']} | Page {s['page']} | Type: {s['content_type']:6s} | Score: {s['score']:.4f}"
        )
        lines.append(f"      Snippet: \"{s['snippet']}\"")
    return "\n".join(lines)

print(format_citations(demo_ans["sources"]))


---
# Phase 3 — Evaluation Framework (MRR-Led)

> Steps 13–18: Test dataset, K-sweep, MRR calculation, answer quality (ROUGE/BLEU), faithfulness, bias/fairness


## Step 13 — Test Questions + Expected Answers Dataset

Define evaluation test cases with expected source PDFs. Edit to match your uploaded PDF knowledge base.


In [ ]:
TEST_DATASET = [
    {
        "id"             : "Q1",
        "question"       : "What is the primary concept described in the document?",
        "expected_answer": "Key educational concepts and principles.",
        "expected_source": pdf_files[0][0] if pdf_files else "textbook.pdf",
        "topic"          : "Overview",
        "content_type"   : "text",
    },
    {
        "id"             : "Q2",
        "question"       : "What data or measurements are presented in any tables?",
        "expected_answer": "Structured tabular measurements and reference values.",
        "expected_source": pdf_files[0][0] if pdf_files else "textbook.pdf",
        "topic"          : "Tables",
        "content_type"   : "table",
    },
    {
        "id"             : "Q3",
        "question"       : "What is shown in the main diagram or figure?",
        "expected_answer": "Visual diagrams illustrating scientific or mathematical structures.",
        "expected_source": pdf_files[0][0] if pdf_files else "textbook.pdf",
        "topic"          : "Images",
        "content_type"   : "image",
    },
]

df_test = pd.DataFrame(TEST_DATASET)
print(f"Test dataset: {len(TEST_DATASET)} questions")
print(df_test[["id", "topic", "content_type", "expected_source", "question"]].to_string(index=False))


## Step 14 — Test Different K Values

Evaluates retrieval and generation across K in {1, 2, 3, 5}.


In [ ]:
K_VALUES = [1, 2, 3, 5]
eval_records = []

print("Running evaluation sweep across K values...")
for item in TEST_DATASET:
    for k in K_VALUES:
        result = generate_rag_answer(
            question=item["question"], top_k=k,
            difficulty="Intermediate", learning_style="Visual",
            temperature=0.0, max_tokens=300
        )
        retrieved_sources = [s["source"] for s in result["sources"]]
        retrieved_types   = [s["content_type"] for s in result["sources"]]
        top1_source       = result["sources"][0]["source"] if result["sources"] else ""
        eval_records.append({
            "qid"              : item["id"],
            "question"         : item["question"],
            "topic"            : item["topic"],
            "expected_ctype"   : item.get("content_type", "text"),
            "k"                : k,
            "expected_source"  : item["expected_source"],
            "expected_answer"  : item["expected_answer"],
            "generated_answer" : result["answer"],
            "context_used"     : result["context_used"],
            "retrieved_sources": retrieved_sources,
            "retrieved_types"  : retrieved_types,
            "top1_source"      : top1_source,
        })
        print(f"  {item['id']} | K={k} | top-1: {top1_source}")

df_eval = pd.DataFrame(eval_records)
print(f"\nEvaluation sweep complete. Total records: {len(df_eval)}")


## Step 15 — Measure Retrieval Quality (MRR as Primary Metric)

- **MRR (Mean Reciprocal Rank)**: Average of `1 / rank` for the first relevant document.
- **Hit Rate @K**: Fraction of queries where the target source appears anywhere in the top-K.
- **Precision @K**: Fraction of retrieved chunks matching the target source.


In [ ]:
def compute_retrieval_metrics(df):
    records = []
    for k_val, grp in df.groupby("k"):
        hits, precisions, mrrs = [], [], []
        for _, row in grp.iterrows():
            srcs, expected = row["retrieved_sources"], row["expected_source"]
            hits.append(int(expected in srcs))
            precisions.append(sum(1 for s in srcs if s == expected) / len(srcs) if srcs else 0)
            rank = next((i + 1 for i, s in enumerate(srcs) if s == expected), None)
            mrrs.append(1.0 / rank if rank else 0.0)
        records.append({
            "K": k_val,
            "MRR": round(np.mean(mrrs), 4),
            "Hit Rate @K": round(np.mean(hits), 4),
            "Precision @K": round(np.mean(precisions), 4),
        })
    return pd.DataFrame(records)

df_retrieval_metrics = compute_retrieval_metrics(df_eval)
print("Retrieval Quality Metrics (MRR highlighted):")
print(df_retrieval_metrics.to_string(index=False))


## Step 16 — Measure Generated-Answer Quality

- **ROUGE-L**: Longest Common Subsequence F1 score.
- **BLEU-1**: Unigram precision against reference answers.


In [ ]:
scorer_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smoother = SmoothingFunction().method1

def answer_quality_metrics(df):
    df = df.copy()
    rouge_scores, bleu_scores = [], []
    for _, row in df.iterrows():
        ref = row["expected_answer"].lower()
        hyp = row["generated_answer"].lower()
        r = scorer_rouge.score(ref, hyp)
        rouge_scores.append(round(r["rougeL"].fmeasure, 4))
        ref_tok = word_tokenize(ref)
        hyp_tok = word_tokenize(hyp)
        b = sentence_bleu([ref_tok], hyp_tok, weights=(1, 0, 0, 0), smoothing_function=smoother)
        bleu_scores.append(round(b, 4))
    df["ROUGE-L"] = rouge_scores
    df["BLEU-1"]  = bleu_scores
    return df

df_eval = answer_quality_metrics(df_eval)
df_answer_metrics = (
    df_eval.groupby("k")[["ROUGE-L", "BLEU-1"]].mean().round(4).reset_index().rename(columns={"k": "K"})
)
print("Answer Quality Metrics:")
print(df_answer_metrics.to_string(index=False))


## Step 17 — Hallucination / Faithfulness Checking

1. **Lexical Faithfulness**: Fraction of generated tokens present in retrieved context.
2. **LLM Self-Judge**: Gemini evaluates whether claims are strictly grounded (0.0–1.0).


In [ ]:
STOP_WORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "and", "or", "but",
    "in", "on", "at", "to", "for", "of", "with", "by", "this", "that", "it", "has",
    "have", "had", "not", "no", "do", "does", "its", "he", "she", "they", "we"
}

def lexical_faithfulness(answer, context):
    ans_tokens = set(word_tokenize(answer.lower())) - STOP_WORDS
    ctx_tokens = set(word_tokenize(context.lower()))
    return round(len(ans_tokens & ctx_tokens) / len(ans_tokens), 4) if ans_tokens else 1.0

FAITH_PROMPT = (
    "You are a strict fact-checker. Given a [CONTEXT] and an [ANSWER], "
    "judge whether every factual claim is explicitly supported by the context.\n\n"
    "[CONTEXT]\n{context}\n\n[ANSWER]\n{answer}\n\n"
    "Respond with JSON only: {{\"score\": float 0.0-1.0, \"reasoning\": \"one sentence\"}}"
)

def llm_faithfulness_score(answer, context):
    prompt = FAITH_PROMPT.format(context=context[:2000], answer=answer)
    cfg = genai_types.GenerateContentConfig(temperature=0.0, max_output_tokens=150)
    resp = client.models.generate_content(model=LLM_MODEL_NAME, contents=prompt, config=cfg)
    raw = resp.text.strip() if resp.text else ""
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return json.loads(m.group()) if m else {"score": 0.5, "reasoning": raw}
    except Exception:
        return {"score": 0.5, "reasoning": raw[:100]}

df_k3 = df_eval[df_eval["k"] == 3].copy().reset_index(drop=True)
lex_scores, llm_scores = [], []
print("Faithfulness evaluation (K=3)...")
for _, row in df_k3.iterrows():
    lex = lexical_faithfulness(row["generated_answer"], row["context_used"])
    lex_scores.append(lex)
    llm_f = llm_faithfulness_score(row["generated_answer"], row["context_used"])
    llm_scores.append(llm_f.get("score", 0.5))
    print(f"  {row['qid']}: lexical={lex:.2f} | LLM={llm_f.get('score', 0.5):.2f}")

df_k3["faithfulness_lexical"] = lex_scores
df_k3["faithfulness_llm"]     = llm_scores
print(f"\nAvg Lexical: {np.mean(lex_scores):.4f} | Avg LLM: {np.mean(llm_scores):.4f}")


## Step 18 — Bias and Fairness Checks

1. **Lexical Scan**: Checks for gendered or stereotyping markers.
2. **LLM Fairness Reviewer**: Evaluates neutrality and objectivity (0.0–1.0).


In [ ]:
GENDERED    = {"he", "she", "his", "her", "him", "hers", "himself", "herself", "man", "men", "woman", "women", "boy", "girl"}
STEREOTYPE  = {"obviously", "naturally", "everyone knows", "of course", "clearly", "typical", "always", "never"}
DEMOGRAPHIC = {"asian", "black", "white", "hispanic", "poor", "rich", "educated", "uneducated", "western", "eastern"}

def lexical_bias_scan(answer):
    tokens = set(word_tokenize(answer.lower()))
    flags  = {
        "gendered"    : [t for t in GENDERED if t in tokens],
        "stereotyping": [t for t in STEREOTYPE if t in tokens],
        "demographic" : [t for t in DEMOGRAPHIC if t in tokens],
    }
    total = sum(len(v) for v in flags.values())
    return {"flags": flags, "total_flags": total, "bias_detected": total > 0}

BIAS_PROMPT = (
    "You are an educational content reviewer specialising in fairness.\n"
    "Evaluate this educational answer for bias, stereotyping, or unfair assumptions.\n\n"
    "[ANSWER]\n{answer}\n\n"
    "Respond JSON only: {{\"bias_score\": float 0.0-1.0, \"issues\": [], \"suggestion\": \"text\"}}"
)

def llm_fairness_score(answer):
    prompt = BIAS_PROMPT.format(answer=answer)
    cfg = genai_types.GenerateContentConfig(temperature=0.0, max_output_tokens=200)
    resp = client.models.generate_content(model=LLM_MODEL_NAME, contents=prompt, config=cfg)
    raw = resp.text.strip() if resp.text else ""
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return json.loads(m.group()) if m else {"bias_score": 0.5, "issues": [], "suggestion": raw}
    except Exception:
        return {"bias_score": 0.5, "issues": [], "suggestion": raw[:100]}

bias_records = []
print("Bias/fairness check (K=3)...")
for _, row in df_k3.iterrows():
    lex  = lexical_bias_scan(row["generated_answer"])
    fair = llm_fairness_score(row["generated_answer"])
    bias_records.append({
        "qid"           : row["qid"],
        "topic"         : row["topic"],
        "lex_flags"     : lex["total_flags"],
        "bias_detected" : lex["bias_detected"],
        "fairness_score": fair.get("bias_score", 0.5),
        "issues"        : ", ".join(fair.get("issues", [])) or "None",
        "suggestion"    : fair.get("suggestion", "None"),
    })
    icon = "ALERT" if lex["bias_detected"] else "OK"
    print(f"  {icon} {row['qid']}: fairness={fair.get('bias_score', 0.5):.2f}")

df_bias = pd.DataFrame(bias_records)
print(f"\nAvg Fairness: {df_bias['fairness_score'].mean():.4f}")


---
## Evaluation Dashboard — MRR-Led Summary

In [ ]:
print("=" * 70)
print(" EDUMENTOR AI v2.0 — FULL EVALUATION REPORT")
print("=" * 70)

# ── Retrieval Quality ──────────────────────────────────────────────────────
print("\nRETRIEVAL QUALITY (all K values):")
print(df_retrieval_metrics.to_string(index=False))

# ── Primary Metric: MRR ────────────────────────────────────────────────────
best_mrr_row = df_retrieval_metrics.loc[df_retrieval_metrics["MRR"].idxmax()]
best_mrr_k   = int(best_mrr_row["K"])
print(f"\n** PRIMARY METRIC — MRR (Mean Reciprocal Rank) **")
print(f"  Best MRR = {best_mrr_row['MRR']:.4f} at K={best_mrr_k}")
print(f"  MRR = 1.0 -> correct source ALWAYS ranked first (perfect)")
print(f"  MRR = 0.5 -> correct source on average at rank 2")

# ── Answer Quality ─────────────────────────────────────────────────────────
print("\nANSWER QUALITY (ROUGE-L and BLEU-1 by K):")
print(df_answer_metrics.to_string(index=False))

# ── Faithfulness ───────────────────────────────────────────────────────────
print("\nFAITHFULNESS (K=3):")
print(df_k3[["qid", "faithfulness_lexical", "faithfulness_llm"]].to_string(index=False))
print(f"  Avg Lexical : {df_k3['faithfulness_lexical'].mean():.4f}")
print(f"  Avg LLM     : {df_k3['faithfulness_llm'].mean():.4f}")

# ── Fairness ───────────────────────────────────────────────────────────────
print("\nFAIRNESS / BIAS (K=3):")
print(df_bias[["qid", "fairness_score", "issues"]].to_string(index=False))
print(f"  Avg Fairness Score: {df_bias['fairness_score'].mean():.4f}")

# ── MRR-led composite scoring ──────────────────────────────────────────────
merged = df_retrieval_metrics.merge(df_answer_metrics, on="K")
merged["composite"] = (
    merged["MRR"]         * 0.40 +
    merged["Hit Rate @K"] * 0.20 +
    merged["ROUGE-L"]     * 0.25 +
    merged["BLEU-1"]      * 0.15
)
best_k = int(merged.loc[merged["composite"].idxmax(), "K"])

print(f"\nCOMPOSITE SCORE (MRR-led: MRR=40% | HitRate=20% | ROUGE-L=25% | BLEU-1=15%):")
for _, row in merged.iterrows():
    bar = '#' * int(row['composite'] * 20)
    marker = " <-- BEST" if int(row["K"]) == best_k else ""
    print(f"  K={int(row['K'])}: [{bar:<20}] {row['composite']:.4f}{marker}")
print(f"\n  OPTIMAL TOP-K = {best_k}")

# ── MRR Dashboard — 3-panel chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("EduMentor AI v2.0 — Evaluation Dashboard (MRR-Led)", fontsize=13, fontweight="bold")

# Panel 1: MRR bar chart (primary metric, best-K in green)
ax = axes[0]
bar_colors = ["#34A853" if int(k) == best_mrr_k else "#4285F4" for k in df_retrieval_metrics["K"]]
bars = ax.bar(df_retrieval_metrics["K"].astype(str), df_retrieval_metrics["MRR"], color=bar_colors, edgecolor="white", width=0.5)
ax.set_title("MRR (Primary Metric)", fontweight="bold", fontsize=11)
ax.set_xlabel("Top-K"); ax.set_ylabel("MRR"); ax.set_ylim(0, 1.15)
ax.axhline(1.0, color="grey", linestyle=":", alpha=0.4, label="Perfect (1.0)")
for bar, val in zip(bars, df_retrieval_metrics["MRR"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}", ha="center", fontsize=11, fontweight="bold")
ax.legend(fontsize=8)

# Panel 2: All retrieval metrics grouped bar
ax2 = axes[1]
k_labels = df_retrieval_metrics["K"].astype(str).tolist()
x = np.arange(len(k_labels))
width = 0.25
for i, (metric, color) in enumerate([("MRR", "#4285F4"), ("Hit Rate @K", "#34A853"), ("Precision @K", "#FBBC04")]):
    ax2.bar(x + i*width, df_retrieval_metrics[metric], width, label=metric, color=color, alpha=0.85, edgecolor="white")
ax2.set_xticks(x + width)
ax2.set_xticklabels(k_labels)
ax2.set_title("All Retrieval Metrics", fontweight="bold", fontsize=11)
ax2.set_xlabel("Top-K"); ax2.set_ylim(0, 1.2); ax2.legend(fontsize=8)

# Panel 3: Composite score (MRR-weighted)
ax3 = axes[2]
comp_colors = ["#34A853" if int(k) == best_k else "#EA4335" for k in merged["K"]]
bars3 = ax3.bar(merged["K"].astype(str), merged["composite"], color=comp_colors, edgecolor="white", width=0.5)
ax3.set_title(f"Composite Score (MRR 40%)\nBest K={best_k}", fontweight="bold", fontsize=11)
ax3.set_xlabel("Top-K"); ax3.set_ylabel("Score"); ax3.set_ylim(0, 1.15)
for bar, val in zip(bars3, merged["composite"]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{val:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig("mrr_dashboard.png", dpi=120, bbox_inches="tight")
plt.show()
print("Dashboard saved: mrr_dashboard.png")

# ── Final model summary ────────────────────────────────────────────────────
print("\n" + "=" * 70)
print(" MODELS USED — EduMentor AI v2.0")
print("=" * 70)
print(f"  PDF text + tables : pdfplumber")
print(f"  PDF images        : PyMuPDF (fitz) + Gemini 3.5 Flash Vision")
print(f"  Embedding         : sentence-transformers/all-MiniLM-L6-v2 (384-dim, 512-tok)")
print(f"  Vector DB         : FAISS (exact cosine, in-memory)")
print(f"  LLM SDK           : google-genai (current official SDK)")
print(f"  LLM model         : gemini-3.5-flash (1M-token context)")
print(f"  Primary metric    : MRR (Mean Reciprocal Rank) — weight 40% in composite")
print("=" * 70)
